In [1]:
#!/usr/bin/env python3
"""
VLSI Design Lab - Assignment 2A
Behavioral Golden Reference Model & Dataset Generator
16x16 Precision-Scalable Systolic Tensor Unit (INT4 / INT8 / INT16)
"""

import os
import numpy as np

# ==============================================================================
# 1. HARDWARE CONFIGURATION PARAMETERS
# ==============================================================================
ARRAY_DIM = 16          # 16x16 Systolic Tensor Unit (256 PEs)
INPUT_BUS_WIDTH = 16    # Physical operand port width (16-bit signed)

# Set ACC_WIDTH to 36 (prevents overflow on -32768 * -32768 * 16 = +2^34)
# or change to 32 if your RTL uses a 32-bit accumulator.
ACC_WIDTH = 36          
ACC_HEX_DIGITS = (ACC_WIDTH + 3) // 4  # 9 hex digits for 36-bit, 8 for 32-bit

# Precision Modes: (mode_1, mode_0) -> (Mode Name, Bit Width, Min Val, Max Val)
PRECISION_MODES = {
    (0, 0): ("INT4",  4,  -8,      7),
    (0, 1): ("INT8",  8,  -128,    127),
    (1, 0): ("INT16", 16, -32768,  32767),
}

# ==============================================================================
# 2. BIT-ACCURATE SIGN EXTENSION & TWO'S COMPLEMENT HELPERS
# ==============================================================================
def extract_and_sign_extend(word_16b: int, active_bits: int) -> int:
    """
    Slices the lower `active_bits` (4, 8, or 16) from a 16-bit physical bus word
    and performs two's-complement sign extension.
    """
    mask = (1 << active_bits) - 1
    raw = int(word_16b) & mask
    sign_bit = 1 << (active_bits - 1)
    if raw & sign_bit:
        return raw - (1 << active_bits)
    return raw


def pack_16bit_sign_extended(val: int, active_bits: int) -> int:
    """
    Clamps/validates `val` within the signed range of `active_bits` and
    sign-extends it to a 16-bit two's-complement unsigned integer (0x0000..0xFFFF).
    """
    min_val = -(1 << (active_bits - 1))
    max_val = (1 << (active_bits - 1)) - 1
    assert min_val <= val <= max_val, f"Value {val} out of INT{active_bits} range [{min_val}, {max_val}]"
    # Sign-extend to 16 bits in two's complement
    return int(val) & 0xFFFF


def to_twos_comp_hex(val: int, bit_width: int) -> str:
    """
    Converts a signed integer `val` into a zero-padded two's-complement hex string
    of width `bit_width` (e.g., 16-bit -> 4 hex chars, 36-bit -> 9 hex chars).
    """
    hex_digits = (bit_width + 3) // 4
    mask = (1 << bit_width) - 1
    unsigned_val = int(val) & mask
    return f"{unsigned_val:0{hex_digits}X}"


def wrap_signed(val: int, bit_width: int) -> int:
    """
    Wraps an arbitrary-precision Python integer to a signed `bit_width` two's-complement integer.
    """
    mask = (1 << bit_width) - 1
    raw = int(val) & mask
    sign_bit = 1 << (bit_width - 1)
    return raw - (1 << bit_width) if (raw & sign_bit) else raw


# ==============================================================================
# 3. BEHAVIORAL GOLDEN MODEL OF 16x16 TENSOR UNIT
# ==============================================================================
def tensor_unit_golden_model(A_bus_16b: np.ndarray, B_bus_16b: np.ndarray, mode_1: int, mode_0: int) -> np.ndarray:
    """
    Behavioral Golden Reference Model for the 16x16 Precision-Scalable Systolic Array.
    
    Inputs:
      - A_bus_16b: (16, 16) array of 16-bit physical bus values (0x0000..0xFFFF)
      - B_bus_16b: (16, 16) array of 16-bit physical bus values (0x0000..0xFFFF)
      - mode_1, mode_0: Global precision mode bits
          00 -> INT4
          01 -> INT8
          10 -> INT16
          11 -> Reserved (Inhibits arithmetic, returns zeros)
    Returns:
      - C_out: (16, 16) array of signed accumulated results wrapped to ACC_WIDTH bits
    """
    if (mode_1, mode_0) == (1, 1):
        # Reserved state: do not initiate arithmetic
        return np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    _, active_bits, _, _ = PRECISION_MODES[(mode_1, mode_0)]

    # Step 1: Each PE slices and sign-extends its incoming 16-bit operand according to mode
    A_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            A_signed[i, j] = extract_and_sign_extend(A_bus_16b[i, j], active_bits)
            B_signed[i, j] = extract_and_sign_extend(B_bus_16b[i, j], active_bits)

    # Step 2: 16x16 Systolic Multiply-Accumulate: C[i,j] = sum_{k=0}^{15} A[i,k] * B[k,j]
    C_acc = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            acc = 0
            for k in range(ARRAY_DIM):
                prod = int(A_signed[i, k]) * int(B_signed[k, j])
                acc = wrap_signed(acc + prod, ACC_WIDTH)
            C_acc[i, j] = acc

    # Cross-check against direct matrix multiplication (when no overflow occurs)
    C_matmul = np.matmul(A_signed, B_signed)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            assert C_acc[i, j] == wrap_signed(C_matmul[i, j], ACC_WIDTH), "Golden model mismatch!"

    return C_acc


# ==============================================================================
# 4. MANDATORY TEST VECTOR GENERATOR (SECTION 13 & 14)
# ==============================================================================
def generate_test_matrices(min_val: int, max_val: int) -> dict:
    """
    Generates all mandatory test cases required by Section 13 and Table A.9
    for a given precision range [min_val, max_val].
    """
    tests = {}

    # 1. Zero Matrix: A = 0, B = 0
    tests["01_zero_matrix"] = (
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 2. Identity Matrix: A = Random Mixed, B = Identity (C should equal A)
    rng_id = np.random.default_rng(seed=100)
    A_id = rng_id.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_id = np.eye(ARRAY_DIM, dtype=np.int64)
    tests["02_identity_matrix"] = (A_id, B_id)

    # 3. Positive Operands Only: [1, max_val]
    rng_pos = np.random.default_rng(seed=101)
    tests["03_positive_operands"] = (
        rng_pos.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_pos.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 4. Negative Operands Only: [min_val, -1]
    rng_neg = np.random.default_rng(seed=102)
    tests["04_negative_operands"] = (
        rng_neg.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_neg.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 5. Mixed-Sign Operands: [min_val, max_val]
    rng_mix = np.random.default_rng(seed=103)
    tests["05_mixed_sign"] = (
        rng_mix.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_mix.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 6. Maximum Positive Values: All elements = max_val (+7, +127, or +32767)
    tests["06_max_positive"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
    )

    # 7. Maximum Negative Values: All elements = min_val (-8, -128, or -32768)
    tests["07_max_negative"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
    )

    # 8, 9, 10. Random Test 1, 2, 3 (Table A.9)
    for idx, seed in enumerate([201, 202, 203], start=1):
        rng_rand = np.random.default_rng(seed=seed)
        tests[f"{7+idx:02d}_random_test_{idx}"] = (
            rng_rand.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
            rng_rand.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        )

    # 11. Cross-Mode Identical Numerical Dataset (Section 14: INT4 -> INT8 -> INT16)
    # Uses values strictly in [-8, +7] so the exact same numbers run in all 3 modes.
    rng_cross = np.random.default_rng(seed=777)
    tests["11_cross_mode_shared"] = (
        rng_cross.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_cross.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    return tests


# ==============================================================================
# 5. EXPORT HEX FILES FOR VERILOG ($readmemh) & SUMMARY LOG
# ==============================================================================
def save_hex_matrix(filepath: str, matrix: np.ndarray, bit_width: int):
    """
    Writes a 16x16 matrix in row-major order (256 lines) as two's-complement hex
    for direct loading in Verilog via $readmemh().
    """
    with open(filepath, "w") as f:
        for i in range(ARRAY_DIM):
            for j in range(ARRAY_DIM):
                f.write(to_twos_comp_hex(int(matrix[i, j]), bit_width) + "\n")


def generate_full_dataset(output_dir: str = "dataset"):
    os.makedirs(output_dir, exist_ok=True)
    report_path = os.path.join(output_dir, "golden_verification_summary.txt")

    with open(report_path, "w") as rep:
        rep.write("=" * 78 + "\n")
        rep.write("16x16 PRECISION-SCALABLE TENSOR UNIT - GOLDEN DATASET REPORT\n")
        rep.write(f"Input Bus Width: {INPUT_BUS_WIDTH}-bit (Sign-Extended) | Accumulator Width: {ACC_WIDTH}-bit\n")
        rep.write("=" * 78 + "\n\n")

        for (m1, m0), (mode_name, bits, min_v, max_v) in PRECISION_MODES.items():
            mode_dir = os.path.join(output_dir, mode_name)
            os.makedirs(mode_dir, exist_ok=True)

            rep.write(f"--- MODE {m1}{m0}: {mode_name} (Signed {bits}-bit, Range [{min_v}, {max_v}]) ---\n")
            tests = generate_test_matrices(min_v, max_v)

            for test_name, (A_val, B_val) in tests.items():
                # Pack A and B into 16-bit sign-extended physical bus format
                A_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                B_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                for r in range(ARRAY_DIM):
                    for c in range(ARRAY_DIM):
                        A_bus[r, c] = pack_16bit_sign_extended(int(A_val[r, c]), bits)
                        B_bus[r, c] = pack_16bit_sign_extended(int(B_val[r, c]), bits)

                # Run Behavioral Golden Model
                C_golden = tensor_unit_golden_model(A_bus, B_bus, m1, m0)

                # Save .hex files for Verilog $readmemh
                file_A = os.path.join(mode_dir, f"{test_name}_A.hex")
                file_B = os.path.join(mode_dir, f"{test_name}_B.hex")
                file_C = os.path.join(mode_dir, f"{test_name}_C_golden.hex")

                save_hex_matrix(file_A, A_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(file_B, B_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(file_C, C_golden, ACC_WIDTH)

                rep.write(
                    f"  [{test_name:22s}] "
                    f"A[0,0]={int(A_val[0,0]):6d} (0x{A_bus[0,0]:04X}) | "
                    f"B[0,0]={int(B_val[0,0]):6d} (0x{B_bus[0,0]:04X}) -> "
                    f"C[0,0]={int(C_golden[0,0]):11d} (0x{to_twos_comp_hex(int(C_golden[0,0]), ACC_WIDTH)})\n"
                )
            rep.write("\n")

    print(f"[SUCCESS] Generated 33 test suites (99 .hex files) in './{output_dir}/'")
    print(f"[REPORT]  Summary written to './{report_path}'")


if __name__ == "__main__":
    generate_full_dataset()

[SUCCESS] Generated 33 test suites (99 .hex files) in './dataset/'
[REPORT]  Summary written to './dataset\golden_verification_summary.txt'


In [3]:
#!/usr/bin/env python3
"""
VLSI Design Lab - Assignment 2A
Behavioral Golden Reference Model & Dataset Generator
16x16 Precision-Scalable Systolic Tensor Unit (INT4 / INT8 / INT16)
"""

import os
import numpy as np

# ==============================================================================
# 1. HARDWARE CONFIGURATION PARAMETERS
# ==============================================================================
ARRAY_DIM = 16          # 16x16 Systolic Tensor Unit (256 PEs)
INPUT_BUS_WIDTH = 16    # Physical operand port width (16-bit signed)

# Set ACC_WIDTH to 36 (prevents overflow on -32768 * -32768 * 16 = +2^34)
# or change to 32 if your RTL uses a 32-bit accumulator.
ACC_WIDTH = 36          
ACC_HEX_DIGITS = (ACC_WIDTH + 3) // 4  # 9 hex digits for 36-bit, 8 for 32-bit

# Precision Modes: (mode_1, mode_0) -> (Mode Name, Bit Width, Min Val, Max Val)
PRECISION_MODES = {
    (0, 0): ("INT4",  4,  -8,      7),
    (0, 1): ("INT8",  8,  -128,    127),
    (1, 0): ("INT16", 16, -32768,  32767),
}

# ==============================================================================
# 2. BIT-ACCURATE SIGN EXTENSION & TWO'S COMPLEMENT HELPERS
# ==============================================================================
def extract_and_sign_extend(word_16b, active_bits):
    """
    Slices the lower `active_bits` (4, 8, or 16) from a 16-bit physical bus word
    and performs two's-complement sign extension.
    """
    mask = (1 << active_bits) - 1
    raw = int(word_16b) & mask
    sign_bit = 1 << (active_bits - 1)
    if raw & sign_bit:
        return raw - (1 << active_bits)
    return raw


def pack_16bit_sign_extended(val, active_bits):
    """
    Clamps/validates `val` within the signed range of `active_bits` and
    sign-extends it to a 16-bit two's-complement unsigned integer (0x0000..0xFFFF).
    """
    min_val = -(1 << (active_bits - 1))
    max_val = (1 << (active_bits - 1)) - 1
    assert min_val <= val <= max_val, f"Value {val} out of INT{active_bits} range [{min_val}, {max_val}]"
    # Sign-extend to 16 bits in two's complement
    return int(val) & 0xFFFF


def to_twos_comp_hex(val, bit_width):
    """
    Converts a signed integer `val` into a zero-padded two's-complement hex string
    of width `bit_width` (e.g., 16-bit -> 4 hex chars, 36-bit -> 9 hex chars).
    """
    hex_digits = (bit_width + 3) // 4
    mask = (1 << bit_width) - 1
    unsigned_val = int(val) & mask
    return f"{unsigned_val:0{hex_digits}X}"


def wrap_signed(val, bit_width):
    """
    Wraps an arbitrary-precision Python integer to a signed `bit_width` two's-complement integer.
    """
    mask = (1 << bit_width) - 1
    raw = int(val) & mask
    sign_bit = 1 << (bit_width - 1)
    return raw - (1 << bit_width) if (raw & sign_bit) else raw


# ==============================================================================
# 3. BEHAVIORAL GOLDEN MODEL OF 16x16 TENSOR UNIT
# ==============================================================================
def tensor_unit_golden_model(A_bus_16b, B_bus_16b, mode_1, mode_0):
    """
    Behavioral Golden Reference Model for the 16x16 Precision-Scalable Systolic Array.
    
    Inputs:
      - A_bus_16b: (16, 16) array of 16-bit physical bus values (0x0000..0xFFFF)
      - B_bus_16b: (16, 16) array of 16-bit physical bus values (0x0000..0xFFFF)
      - mode_1, mode_0: Global precision mode bits
          00 -> INT4
          01 -> INT8
          10 -> INT16
          11 -> Reserved (Inhibits arithmetic, returns zeros)
    Returns:
      - C_out: (16, 16) array of signed accumulated results wrapped to ACC_WIDTH bits
    """
    if (mode_1, mode_0) == (1, 1):
        # Reserved state: do not initiate arithmetic
        return np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    _, active_bits, _, _ = PRECISION_MODES[(mode_1, mode_0)]

    # Step 1: Each PE slices and sign-extends its incoming 16-bit operand according to mode
    A_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            A_signed[i, j] = extract_and_sign_extend(A_bus_16b[i, j], active_bits)
            B_signed[i, j] = extract_and_sign_extend(B_bus_16b[i, j], active_bits)

    # Step 2: 16x16 Systolic Multiply-Accumulate: C[i,j] = sum_{k=0}^{15} A[i,k] * B[k,j]
    C_acc = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            acc = 0
            for k in range(ARRAY_DIM):
                prod = int(A_signed[i, k]) * int(B_signed[k, j])
                acc = wrap_signed(acc + prod, ACC_WIDTH)
            C_acc[i, j] = acc

    # Cross-check against direct matrix multiplication (when no overflow occurs)
    C_matmul = np.matmul(A_signed, B_signed)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            assert C_acc[i, j] == wrap_signed(C_matmul[i, j], ACC_WIDTH), "Golden model mismatch!"

    return C_acc


# ==============================================================================
# 4. MANDATORY TEST VECTOR GENERATOR (SECTION 13 & 14)
# ==============================================================================
def generate_test_matrices(min_val, max_val):
    """
    Generates all mandatory test cases required by Section 13 and Table A.9
    for a given precision range [min_val, max_val].
    """
    tests = {}

    # 1. Zero Matrix: A = 0, B = 0
    tests["01_zero_matrix"] = (
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 2. Identity Matrix: A = Random Mixed, B = Identity (C should equal A)
    rng_id = np.random.default_rng(seed=100)
    A_id = rng_id.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_id = np.eye(ARRAY_DIM, dtype=np.int64)
    tests["02_identity_matrix"] = (A_id, B_id)

    # 3. Positive Operands Only: [1, max_val]
    rng_pos = np.random.default_rng(seed=101)
    tests["03_positive_operands"] = (
        rng_pos.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_pos.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 4. Negative Operands Only: [min_val, -1]
    rng_neg = np.random.default_rng(seed=102)
    tests["04_negative_operands"] = (
        rng_neg.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_neg.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 5. Mixed-Sign Operands: [min_val, max_val]
    rng_mix = np.random.default_rng(seed=103)
    tests["05_mixed_sign"] = (
        rng_mix.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_mix.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # 6. Maximum Positive Values: All elements = max_val (+7, +127, or +32767)
    tests["06_max_positive"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
    )

    # 7. Maximum Negative Values: All elements = min_val (-8, -128, or -32768)
    tests["07_max_negative"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
    )

    # 8, 9, 10. Random Test 1, 2, 3 (Table A.9)
    for idx, seed in enumerate([201, 202, 203], start=1):
        rng_rand = np.random.default_rng(seed=seed)
        tests[f"{7+idx:02d}_random_test_{idx}"] = (
            rng_rand.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
            rng_rand.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        )

    # 11. Cross-Mode Identical Numerical Dataset (Section 14: INT4 -> INT8 -> INT16)
    # Uses values strictly in [-8, +7] so the exact same numbers run in all 3 modes.
    rng_cross = np.random.default_rng(seed=777)
    tests["11_cross_mode_shared"] = (
        rng_cross.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng_cross.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    return tests


# ==============================================================================
# 5. EXPORT HEX FILES FOR VERILOG ($readmemh) & SUMMARY LOG
# ==============================================================================
def save_hex_matrix(filepath, matrix, bit_width):
    """
    Writes a 16x16 matrix in row-major order (256 lines) as two's-complement hex
    for direct loading in Verilog via $readmemh().
    """
    with open(filepath, "w") as f:
        for i in range(ARRAY_DIM):
            for j in range(ARRAY_DIM):
                f.write(to_twos_comp_hex(int(matrix[i, j]), bit_width) + "\n")


def generate_full_dataset(output_dir="dataset"):
    os.makedirs(output_dir, exist_ok=True)
    report_path = os.path.join(output_dir, "golden_verification_summary.txt")

    with open(report_path, "w") as rep:
        rep.write("=" * 78 + "\n")
        rep.write("16x16 PRECISION-SCALABLE TENSOR UNIT - GOLDEN DATASET REPORT\n")
        rep.write(f"Input Bus Width: {INPUT_BUS_WIDTH}-bit (Sign-Extended) | Accumulator Width: {ACC_WIDTH}-bit\n")
        rep.write("=" * 78 + "\n\n")

        for (m1, m0), (mode_name, bits, min_v, max_v) in PRECISION_MODES.items():
            mode_dir = os.path.join(output_dir, mode_name)
            os.makedirs(mode_dir, exist_ok=True)

            rep.write(f"--- MODE {m1}{m0}: {mode_name} (Signed {bits}-bit, Range [{min_v}, {max_v}]) ---\n")
            tests = generate_test_matrices(min_v, max_v)

            for test_name, (A_val, B_val) in tests.items():
                # Pack A and B into 16-bit sign-extended physical bus format
                A_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                B_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                for r in range(ARRAY_DIM):
                    for c in range(ARRAY_DIM):
                        A_bus[r, c] = pack_16bit_sign_extended(int(A_val[r, c]), bits)
                        B_bus[r, c] = pack_16bit_sign_extended(int(B_val[r, c]), bits)

                # Run Behavioral Golden Model
                C_golden = tensor_unit_golden_model(A_bus, B_bus, m1, m0)

                # Save .hex files for Verilog $readmemh
                file_A = os.path.join(mode_dir, f"{test_name}_A.hex")
                file_B = os.path.join(mode_dir, f"{test_name}_B.hex")
                file_C = os.path.join(mode_dir, f"{test_name}_C_golden.hex")

                save_hex_matrix(file_A, A_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(file_B, B_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(file_C, C_golden, ACC_WIDTH)

                rep.write(
                    f"  [{test_name:22s}] "
                    f"A[0,0]={int(A_val[0,0]):6d} (0x{A_bus[0,0]:04X}) | "
                    f"B[0,0]={int(B_val[0,0]):6d} (0x{B_bus[0,0]:04X}) -> "
                    f"C[0,0]={int(C_golden[0,0]):11d} (0x{to_twos_comp_hex(int(C_golden[0,0]), ACC_WIDTH)})\n"
                )
            rep.write("\n")

    print(f"[SUCCESS] Generated 33 test suites (99 .hex files) in './{output_dir}/'")
    print(f"[REPORT]  Summary written to './{report_path}'")


if __name__ == "__main__":
    generate_full_dataset()

[SUCCESS] Generated 33 test suites (99 .hex files) in './dataset/'
[REPORT]  Summary written to './dataset\golden_verification_summary.txt'


In [5]:
import os
import numpy as np

# Basic hardware settings
ARRAY_DIM = 16          # The array has 16 rows and 16 columns
INPUT_BUS_WIDTH = 16    # Each input is carried on a 16-bit bus
ACC_WIDTH = 36          # Width used to store each accumulated result

# The mode bits select the number of bits used from each input.
# Each entry contains: mode name, active input bits, minimum value, maximum value.
PRECISION_MODES = {
    (0, 0): ("INT4", 4, -8, 7),
    (0, 1): ("INT8", 8, -128, 127),
    (1, 0): ("INT16", 16, -32768, 32767),
}


def extract_and_sign_extend(word, active_bits):
    # Make a mask with the lowest active_bits set to 1.
    mask = (1 << active_bits) - 1

    # Keep only the active bits from the 16-bit input word.
    raw = int(word) & mask

    # In a signed number, the highest active bit indicates a negative value.
    sign_bit = 1 << (active_bits - 1)
    if raw & sign_bit:
        # Convert a negative two's-complement value to a Python negative integer.
        return raw - (1 << active_bits)

    return raw


def pack_16bit_sign_extended(val, active_bits):
    # Check that the value can be represented in the selected signed precision.
    min_val = -(1 << (active_bits - 1))
    max_val = (1 << (active_bits - 1)) - 1
    if not min_val <= val <= max_val:
        raise ValueError(f"{val} is outside the INT{active_bits} range")

    # Keep the lower 16 bits. Negative values become 16-bit two's-complement values.
    return int(val) & 0xFFFF


def to_twos_comp_hex(val, bit_width):
    # One hexadecimal digit represents 4 bits, so calculate the number of digits needed.
    hex_digits = (bit_width + 3) // 4

    # Keep only bit_width bits, then format as uppercase hex with leading zeros.
    mask = (1 << bit_width) - 1
    unsigned_val = int(val) & mask
    return f"{unsigned_val:0{hex_digits}X}"


def wrap_signed(val, bit_width):
    # Simulate a fixed-width hardware result by discarding bits above bit_width.
    mask = (1 << bit_width) - 1
    raw = int(val) & mask

    # If the top bit is 1, interpret the value as a negative two's-complement number.
    sign_bit = 1 << (bit_width - 1)
    if raw & sign_bit:
        return raw - (1 << bit_width)
    return raw


def tensor_unit_golden_model(A_bus, B_bus, mode_1, mode_0):
    # Mode 11 is reserved, so return an all-zero output matrix for that mode.
    if (mode_1, mode_0) == (1, 1):
        return np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    # Find how many low bits should be used for the selected mode.
    _, active_bits, _, _ = PRECISION_MODES[(mode_1, mode_0)]

    # Decode each bus word into a signed number using the selected precision.
    A_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            A_signed[i, j] = extract_and_sign_extend(A_bus[i, j], active_bits)
            B_signed[i, j] = extract_and_sign_extend(B_bus[i, j], active_bits)

    # Calculate C = A x B. Each output element is the sum of 16 products.
    C = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            acc = 0
            for k in range(ARRAY_DIM):
                product = int(A_signed[i, k]) * int(B_signed[k, j])

                # Apply the 36-bit limit after each addition, as fixed-width hardware would.
                acc = wrap_signed(acc + product, ACC_WIDTH)
            C[i, j] = acc

    return C


def generate_test_matrices(min_val, max_val):
    # Store each test as a pair of input matrices: (A, B).
    tests = {}

    # Test 1: zero inputs should produce a zero output matrix.
    tests["01_zero_matrix"] = (
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 2: multiplying by the identity matrix should return A unchanged.
    rng = np.random.default_rng(seed=100)
    tests["02_identity_matrix"] = (
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.eye(ARRAY_DIM, dtype=np.int64),
    )

    # Test 3: use only positive operands.
    rng = np.random.default_rng(seed=101)
    tests["03_positive_operands"] = (
        rng.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 4: use only negative operands.
    rng = np.random.default_rng(seed=102)
    tests["04_negative_operands"] = (
        rng.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 5: mix positive and negative values.
    rng = np.random.default_rng(seed=103)
    tests["05_mixed_sign"] = (
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 6: every input element is the largest positive value for this mode.
    tests["06_max_positive"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
    )

    # Test 7: every input element is the most negative value for this mode.
    tests["07_max_negative"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
    )

    # Tests 8-10: three repeatable random datasets with different seeds.
    for idx, seed in enumerate((201, 202, 203), start=1):
        rng = np.random.default_rng(seed=seed)
        tests[f"{7 + idx:02d}_random_test_{idx}"] = (
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        )

    # Test 11: use the same small values in INT4, INT8 and INT16 modes for comparison.
    rng = np.random.default_rng(seed=777)
    tests["11_cross_mode_shared"] = (
        rng.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    return tests


def save_hex_matrix(filepath, matrix, bit_width):
    # Write one matrix value per line, in row-major order, for Verilog $readmemh.
    with open(filepath, "w") as file:
        for row in range(ARRAY_DIM):
            for col in range(ARRAY_DIM):
                value = to_twos_comp_hex(matrix[row, col], bit_width)
                file.write(value + "\n")


def generate_full_dataset(output_dir="dataset"):
    # Create the output folder if it does not already exist.
    os.makedirs(output_dir, exist_ok=True)
    report_path = os.path.join(output_dir, "golden_verification_summary.txt")

    # Write a short summary of the values generated for each test case.
    with open(report_path, "w") as report:
        report.write("16x16 Systolic Tensor Unit - Golden Model Results\n")
        report.write(f"Input width: {INPUT_BUS_WIDTH} bits; accumulator width: {ACC_WIDTH} bits\n\n")

        # Generate the same set of tests for INT4, INT8 and INT16.
        for (mode_1, mode_0), (mode_name, bits, min_val, max_val) in PRECISION_MODES.items():
            mode_dir = os.path.join(output_dir, mode_name)
            os.makedirs(mode_dir, exist_ok=True)
            report.write(f"Mode {mode_1}{mode_0}: {mode_name}, range {min_val} to {max_val}\n")

            tests = generate_test_matrices(min_val, max_val)
            for test_name, (A, B) in tests.items():
                # Convert the test matrices into the 16-bit format used on the input bus.
                A_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                B_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                for row in range(ARRAY_DIM):
                    for col in range(ARRAY_DIM):
                        A_bus[row, col] = pack_16bit_sign_extended(int(A[row, col]), bits)
                        B_bus[row, col] = pack_16bit_sign_extended(int(B[row, col]), bits)

                # Calculate the expected output using the golden model.
                C = tensor_unit_golden_model(A_bus, B_bus, mode_1, mode_0)

                # Save A, B and the expected C matrix in files for RTL testbench use.
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_A.hex"), A_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_B.hex"), B_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_C_golden.hex"), C, ACC_WIDTH)

                # Record the first element as a quick summary of this test case.
                report.write(
                    f"  {test_name}: A[0,0]={int(A[0, 0])}, "
                    f"B[0,0]={int(B[0, 0])}, C[0,0]={int(C[0, 0])}\n"
                )
            report.write("\n")

    print(f"Generated 99 HEX files in '{output_dir}/'")
    print(f"Summary saved to '{report_path}'")


# Run dataset generation when this file is executed directly.
if __name__ == "__main__":
    generate_full_dataset()


Generated 99 HEX files in 'dataset/'
Summary saved to 'dataset\golden_verification_summary.txt'


In [6]:
import os
import numpy as np

# Basic hardware settings
ARRAY_DIM = 16          # The array has 16 rows and 16 columns
INPUT_BUS_WIDTH = 16    # Each input is carried on a 16-bit bus
ACC_WIDTH = 36          # Width used to store each accumulated result

# The mode bits select the number of bits used from each input.
# Each entry contains: mode name, active input bits, minimum value, maximum value.
PRECISION_MODES = {
    (0, 0): ("INT4", 4, -8, 7),
    (0, 1): ("INT8", 8, -128, 127),
    (1, 0): ("INT16", 16, -32768, 32767),
}


def extract_and_sign_extend(word, active_bits):
    # Make a mask with the lowest active_bits set to 1.
    mask = (1 << active_bits) - 1

    # Keep only the active bits from the 16-bit input word.
    raw = int(word) & mask

    # In a signed number, the highest active bit indicates a negative value.
    sign_bit = 1 << (active_bits - 1)
    if raw & sign_bit:
        # Convert a negative two's-complement value to a Python negative integer.
        return raw - (1 << active_bits)

    return raw


def pack_16bit_sign_extended(val, active_bits):
    # Check that the value can be represented in the selected signed precision.
    min_val = -(1 << (active_bits - 1))
    max_val = (1 << (active_bits - 1)) - 1
    if not min_val <= val <= max_val:
        raise ValueError(f"{val} is outside the INT{active_bits} range")

    # Keep the lower 16 bits. Negative values become 16-bit two's-complement values.
    return int(val) & 0xFFFF


def to_twos_comp_hex(val, bit_width):
    # One hexadecimal digit represents 4 bits, so calculate the number of digits needed.
    hex_digits = (bit_width + 3) // 4

    # Keep only bit_width bits, then format as uppercase hex with leading zeros.
    mask = (1 << bit_width) - 1
    unsigned_val = int(val) & mask
    return f"{unsigned_val:0{hex_digits}X}"


def wrap_signed(val, bit_width):
    # Simulate a fixed-width hardware result by discarding bits above bit_width.
    mask = (1 << bit_width) - 1
    raw = int(val) & mask

    # If the top bit is 1, interpret the value as a negative two's-complement number.
    sign_bit = 1 << (bit_width - 1)
    if raw & sign_bit:
        return raw - (1 << bit_width)
    return raw


def tensor_unit_golden_model(A_bus, B_bus, mode_1, mode_0):
    # Mode 11 is reserved, so return an all-zero output matrix for that mode.
    if (mode_1, mode_0) == (1, 1):
        return np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)

    # Find how many low bits should be used for the selected mode.
    _, active_bits, _, _ = PRECISION_MODES[(mode_1, mode_0)]

    # Decode each bus word into a signed number using the selected precision.
    A_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    B_signed = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            A_signed[i, j] = extract_and_sign_extend(A_bus[i, j], active_bits)
            B_signed[i, j] = extract_and_sign_extend(B_bus[i, j], active_bits)

    # Calculate C = A x B. Each output element is the sum of 16 products.
    C = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64)
    for i in range(ARRAY_DIM):
        for j in range(ARRAY_DIM):
            acc = 0
            for k in range(ARRAY_DIM):
                product = int(A_signed[i, k]) * int(B_signed[k, j])

                # Apply the 36-bit limit after each addition, as fixed-width hardware would.
                acc = wrap_signed(acc + product, ACC_WIDTH)
            C[i, j] = acc

    return C


def generate_test_matrices(min_val, max_val):
    # Store each test as a pair of input matrices: (A, B).
    tests = {}

    # Test 1: zero inputs should produce a zero output matrix.
    tests["01_zero_matrix"] = (
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 2: multiplying by the identity matrix should return A unchanged.
    rng = np.random.default_rng(seed=100)
    tests["02_identity_matrix"] = (
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        np.eye(ARRAY_DIM, dtype=np.int64),
    )

    # Test 3: use only positive operands.
    rng = np.random.default_rng(seed=101)
    tests["03_positive_operands"] = (
        rng.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(1, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 4: use only negative operands.
    rng = np.random.default_rng(seed=102)
    tests["04_negative_operands"] = (
        rng.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(min_val, 0, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 5: mix positive and negative values.
    rng = np.random.default_rng(seed=103)
    tests["05_mixed_sign"] = (
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Test 6: every input element is the largest positive value for this mode.
    tests["06_max_positive"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), max_val, dtype=np.int64),
    )

    # Test 7: every input element is the most negative value for this mode.
    tests["07_max_negative"] = (
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
        np.full((ARRAY_DIM, ARRAY_DIM), min_val, dtype=np.int64),
    )

    # Tests 8-10: three repeatable random datasets with different seeds.
    for idx, seed in enumerate((201, 202, 203), start=1):
        rng = np.random.default_rng(seed=seed)
        tests[f"{7 + idx:02d}_random_test_{idx}"] = (
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        )

    # Test 11: use the same small values in INT4, INT8 and INT16 modes for comparison.
    rng = np.random.default_rng(seed=777)
    tests["11_cross_mode_shared"] = (
        rng.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        rng.integers(-8, 8, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
    )

    # Tests 12-100: add more random datasets to reach 100 tests per precision mode.
    # Fixed, different seeds make every test repeatable across runs.
    for test_number, random_number in enumerate(range(4, 93), start=12):
        rng = np.random.default_rng(seed=200 + random_number)
        tests[f"{test_number:02d}_random_test_{random_number}"] = (
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
            rng.integers(min_val, max_val + 1, size=(ARRAY_DIM, ARRAY_DIM), dtype=np.int64),
        )

    return tests


def save_hex_matrix(filepath, matrix, bit_width):
    # Write one matrix value per line, in row-major order, for Verilog $readmemh.
    with open(filepath, "w") as file:
        for row in range(ARRAY_DIM):
            for col in range(ARRAY_DIM):
                value = to_twos_comp_hex(matrix[row, col], bit_width)
                file.write(value + "\n")


def generate_full_dataset(output_dir="dataset"):
    # Create the output folder if it does not already exist.
    os.makedirs(output_dir, exist_ok=True)
    report_path = os.path.join(output_dir, "golden_verification_summary.txt")

    # Write a short summary of the values generated for each test case.
    with open(report_path, "w") as report:
        report.write("16x16 Systolic Tensor Unit - Golden Model Results\n")
        report.write(f"Input width: {INPUT_BUS_WIDTH} bits; accumulator width: {ACC_WIDTH} bits\n")
        report.write("Test cases per precision mode: 100\n")
        report.write("Total test cases across all modes: 300\n\n")

        # Generate the same set of tests for INT4, INT8 and INT16.
        for (mode_1, mode_0), (mode_name, bits, min_val, max_val) in PRECISION_MODES.items():
            mode_dir = os.path.join(output_dir, mode_name)
            os.makedirs(mode_dir, exist_ok=True)
            report.write(f"Mode {mode_1}{mode_0}: {mode_name}, range {min_val} to {max_val}\n")

            tests = generate_test_matrices(min_val, max_val)
            for test_name, (A, B) in tests.items():
                # Convert the test matrices into the 16-bit format used on the input bus.
                A_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                B_bus = np.zeros((ARRAY_DIM, ARRAY_DIM), dtype=np.uint16)
                for row in range(ARRAY_DIM):
                    for col in range(ARRAY_DIM):
                        A_bus[row, col] = pack_16bit_sign_extended(int(A[row, col]), bits)
                        B_bus[row, col] = pack_16bit_sign_extended(int(B[row, col]), bits)

                # Calculate the expected output using the golden model.
                C = tensor_unit_golden_model(A_bus, B_bus, mode_1, mode_0)

                # Save A, B and the expected C matrix in files for RTL testbench use.
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_A.hex"), A_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_B.hex"), B_bus, INPUT_BUS_WIDTH)
                save_hex_matrix(os.path.join(mode_dir, f"{test_name}_C_golden.hex"), C, ACC_WIDTH)

                # Record the first element as a quick summary of this test case.
                report.write(
                    f"  {test_name}: A[0,0]={int(A[0, 0])}, "
                    f"B[0,0]={int(B[0, 0])}, C[0,0]={int(C[0, 0])}\n"
                )
            report.write("\n")

    print(f"Generated 300 test cases (100 per precision mode) in '{output_dir}/'")
    print(f"Generated 900 HEX files in '{output_dir}/' (A, B and golden C for each test)")
    print(f"Summary saved to '{report_path}'")


# Run dataset generation when this file is executed directly.
if __name__ == "__main__":
    generate_full_dataset()


Generated 300 test cases (100 per precision mode) in 'dataset/'
Generated 900 HEX files in 'dataset/' (A, B and golden C for each test)
Summary saved to 'dataset\golden_verification_summary.txt'
